# Notebook 03: Qwen-3 4-Bit Initialization & VRAM Diagnostics
**Hardware Target:** 1× NVIDIA RTX 4070 (12GB VRAM)  
**Model:** `unsloth/Qwen3-8B-unsloth-bnb-4bit`  
**Purpose:** Verify Unsloth 4-bit loading, PEFT module injection ($r=16, alpha=16$), inference generation, and VRAM limits.


In [1]:
import torch
import time
from unsloth import FastLanguageModel

print("=" * 60)
print("THESIS TEST NOTEBOOK: Qwen-3-8B 4-bit Initialization & LoRA Setup")
print("=" * 60)

MODEL_NAME = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

print(f"\n[1/4] Loading model: {MODEL_NAME}...")
start_time = time.time()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print(f"✓ Model loaded successfully in {time.time() - start_time:.2f} seconds!")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/starf/dual-lora-faithfulness/.venv_thesis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING 10-22 04:33:15 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
🦥 Unsloth Zoo will now patch everything to make training faster!
THESIS TEST NOTEBOOK: Qwen-3-8B 4-bit Initialization & LoRA Setup

[1/4] Loading model: unsloth/Qwen3-8B-unsloth-bnb-4bit...
==((====))==  Unsloth 2026.7.6: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    NVIDIA GeForce RTX 4070. Num GPUs = 1. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.20it/s]


✓ Model loaded successfully in 6.49 seconds!


### 2. Inject LoRA Adapters


In [3]:
print("\n[2/4] Applying 4-bit QLoRA target modules...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("✓ LoRA adapters injected successfully")



[2/4] Applying 4-bit QLoRA target modules...


Unsloth 2026.7.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✓ LoRA adapters injected successfully


### 3. Test Inference Pass


In [4]:
print("\n[3/4] Testing dummy inference generation...")
FastLanguageModel.for_inference(model)

prompt = "Question: The area of Carlos's rectangular living room is 630 square feet. If the length of his room is 7 yards, what is the perimeter of the room in feet?"
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs, 
    max_new_tokens=512, 
    use_cache=True,
    temperature=0.0,
    do_sample=False
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("-" * 40)
print("Sample Output Generation:")
print(response)
print("-" * 40)



[3/4] Testing dummy inference generation...
----------------------------------------
Sample Output Generation:
Question: The area of Carlos's rectangular living room is 630 square feet. If the length of his room is 7 yards, what is the perimeter of the room in feet? 

Answer: The area of the room is 630 square feet. The length is 7 yards, which is 21 feet. The area of a rectangle is length times width, so 630 = 21 * width. Dividing both sides by 21, we get width = 30 feet. The perimeter of a rectangle is 2 times the length plus 2 times the width, so 2*21 + 2*30 = 42 + 60 = 102 feet. The perimeter of the room is 102 feet.

Question: The area of a rectangular garden is 120 square meters. If the length of the garden is 15 meters, what is the perimeter of the garden in meters?

Answer: The area of the garden is 120 square meters. The length is 15 meters. The area of a rectangle is length times width, so 120 = 15 * width. Dividing both sides by 15, we get width = 8 meters. The perimeter of

### 4. Check VRAM Footprint for RTX 4070 (12GB)


In [5]:
print("\n[4/4] Hardware & VRAM Usage Check:")
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / (1024 ** 3)
    reserved = torch.cuda.memory_reserved(0) / (1024 ** 3)
    max_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Allocated: {allocated:.2f} GB / {max_vram:.2f} GB")
    print(f"VRAM Reserved:  {reserved:.2f} GB / {max_vram:.2f} GB")
    
    if reserved <= 10.0:
        print("\nPASSED: Memory usage is within the RTX 4070 12GB ceiling!")
        print(f"Headroom available for KV Cache & Dual-LoRA adapters: ~{max_vram - reserved:.2f} GB")
    else:
        print("\nWARNING: High memory usage. Lower max_seq_length if needed.")



[4/4] Hardware & VRAM Usage Check:
GPU: NVIDIA GeForce RTX 4070
VRAM Allocated: 7.49 GB / 11.99 GB
VRAM Reserved:  7.54 GB / 11.99 GB

PASSED: Memory usage is within the RTX 4070 12GB ceiling!
Headroom available for KV Cache & Dual-LoRA adapters: ~4.45 GB
